# PyTorch 手写数字识别（MNIST）项目

本文演示如何使用 **PyTorch** 构建一个卷积神经网络（CNN）来识别 MNIST 手写数字数据集。

**内容包括：**
- 环境检测
- 数据集加载
- CNN模型构建
- 模型训练
- 模型测试
- 模型预测

MNIST 是深度学习入门最经典的数据集之一。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

## 检查 PyTorch 环境

In [ ]:
print('PyTorch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 加载 MNIST 数据集

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

## 查看数据样例

In [ ]:
image, label = train_dataset[0]

plt.imshow(image.squeeze(), cmap='gray')
plt.title(f'Label: {label}')
plt.show()

## 构建 CNN 模型

In [ ]:
class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)

        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))

        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)

        x = torch.relu(self.fc1(x))
        x = self.fc2(x)

        return x

## 初始化模型

In [ ]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model

## 模型训练

In [ ]:
epochs = 5

for epoch in range(epochs):

    total_loss = 0

    for data, target in train_loader:

        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(data)

        loss = criterion(output, target)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {total_loss:.4f}')

## 模型测试

In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for data, target in test_loader:

        data = data.to(device)

        outputs = model(data)

        _, predicted = torch.max(outputs.data, 1)

        total += target.size(0)
        correct += (predicted.cpu() == target).sum().item()

print('Accuracy:', correct / total)

## 模型预测示例

In [ ]:
dataiter = iter(test_loader)

images, labels = next(dataiter)

image = images[0].to(device)

output = model(image.unsqueeze(0))

pred = output.argmax(dim=1)

plt.imshow(images[0].squeeze(), cmap='gray')
plt.title(f'Prediction: {pred.item()}')
plt.show()

## 保存模型

In [ ]:
torch.save(model.state_dict(), 'mnist_cnn.pth')

## 总结

本文实现了一个完整的 PyTorch MNIST 手写数字识别流程：

1. 数据加载
2. CNN模型构建
3. 模型训练
4. 模型测试
5. 模型预测

该模型通常可以达到 **98%+ 的识别准确率**。

这是深度学习入门最经典的项目之一。